# CodeAlpha – Credit Scoring Model
**Task 1 | Machine Learning Internship**

This notebook loads the Credit Score Classification dataset, cleans the data, trains and compares Logistic Regression and Random Forest pipelines, evaluates them on a stratified validation split, saves the best model, and predicts the test dataset.

## 1. Import libraries

In [3]:
import os
import re
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

## 2. Load dataset

In [4]:
DATA_DIR = "/kaggle/input/datasets/parisrohan/credit-score-classification"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
display(train_df.head())

Train shape: (100000, 28)
Test shape: (50000, 27)


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


## 3. Inspect target and missing values

In [5]:
print("Target distribution:")
display(train_df["Credit_Score"].value_counts())

print("\\nMissing values in train:")
display(train_df.isna().sum().sort_values(ascending=False).head(15))

print("\\nMissing values in test:")
display(test_df.isna().sum().sort_values(ascending=False).head(15))

Target distribution:


Credit_Score
Standard    53174
Poor        28998
Good        17828
Name: count, dtype: int64

\nMissing values in train:


Monthly_Inhand_Salary      15002
Type_of_Loan               11408
Name                        9985
Credit_History_Age          9030
Num_of_Delayed_Payment      7002
Amount_invested_monthly     4479
Num_Credit_Inquiries        1965
Monthly_Balance             1200
Month                          0
Age                            0
Customer_ID                    0
ID                             0
Interest_Rate                  0
Num_Credit_Card                0
Num_Bank_Accounts              0
dtype: int64

\nMissing values in test:


Monthly_Inhand_Salary      7498
Type_of_Loan               5704
Name                       5015
Credit_History_Age         4470
Num_of_Delayed_Payment     3498
Amount_invested_monthly    2271
Num_Credit_Inquiries       1035
Monthly_Balance             562
Customer_ID                   0
Month                         0
ID                            0
Num_Credit_Card               0
Num_Bank_Accounts             0
Annual_Income                 0
Occupation                    0
dtype: int64

## 4. Prepare features and clean data

The ID, customer identifier, name, and SSN fields are excluded from model training. Numeric-looking columns containing underscores are converted to numeric values. Implausible values are changed to missing values so the preprocessing pipeline can impute them using training data only.

In [6]:
DROP_COLS = ["Credit_Score", "ID", "Customer_ID", "Name", "SSN"]

X_raw = train_df.drop(columns=["Credit_Score"] + [c for c in DROP_COLS[1:] if c in train_df.columns]).copy()
y = train_df["Credit_Score"].copy()

X_test_raw = test_df.drop(columns=[c for c in ["ID", "Customer_ID", "Name", "SSN"] if c in test_df.columns]).copy()

numeric_text_cols = [
    "Age", "Annual_Income", "Num_of_Loan", "Num_of_Delayed_Payment",
    "Changed_Credit_Limit", "Outstanding_Debt",
    "Amount_invested_monthly", "Monthly_Balance"
]

def clean_features(df):
    df = df.copy()

    # Convert numeric values stored as strings (some contain underscores).
    for col in numeric_text_cols:
        if col in df.columns:
            cleaned = df[col].astype("string").str.replace("_", "", regex=False)
            df[col] = pd.to_numeric(cleaned, errors="coerce")

    # Convert all expected numeric columns to numeric dtype.
    expected_numeric = [
        "Age", "Annual_Income", "Monthly_Inhand_Salary",
        "Num_Bank_Accounts", "Num_Credit_Card", "Interest_Rate",
        "Num_of_Loan", "Delay_from_due_date", "Num_of_Delayed_Payment",
        "Changed_Credit_Limit", "Num_Credit_Inquiries",
        "Outstanding_Debt", "Credit_Utilization_Ratio",
        "Total_EMI_per_month", "Amount_invested_monthly", "Monthly_Balance"
    ]
    for col in expected_numeric:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col].astype("string").str.replace("_", "", regex=False),
                errors="coerce"
            )

    # Remove implausible values; imputation happens later inside the pipeline.
    if "Age" in df.columns:
        df.loc[(df["Age"] < 14) | (df["Age"] > 100), "Age"] = np.nan
    if "Num_Bank_Accounts" in df.columns:
        df.loc[(df["Num_Bank_Accounts"] < 0) | (df["Num_Bank_Accounts"] > 20), "Num_Bank_Accounts"] = np.nan
    if "Num_Credit_Card" in df.columns:
        df.loc[(df["Num_Credit_Card"] < 0) | (df["Num_Credit_Card"] > 20), "Num_Credit_Card"] = np.nan
    if "Interest_Rate" in df.columns:
        df.loc[(df["Interest_Rate"] < 0) | (df["Interest_Rate"] > 100), "Interest_Rate"] = np.nan
    if "Num_Credit_Inquiries" in df.columns:
        df.loc[(df["Num_Credit_Inquiries"] < 0) | (df["Num_Credit_Inquiries"] > 100), "Num_Credit_Inquiries"] = np.nan
    if "Num_of_Loan" in df.columns:
        df.loc[(df["Num_of_Loan"] < 0) | (df["Num_of_Loan"] > 20), "Num_of_Loan"] = np.nan
    if "Num_of_Delayed_Payment" in df.columns:
        df.loc[(df["Num_of_Delayed_Payment"] < 0) | (df["Num_of_Delayed_Payment"] > 100), "Num_of_Delayed_Payment"] = np.nan
    if "Delay_from_due_date" in df.columns:
        df.loc[df["Delay_from_due_date"] < 0, "Delay_from_due_date"] = np.nan

    # Replace common missing-value markers in text columns.
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].replace(["_", "_______", ""], np.nan)

    return df

X = clean_features(X_raw)
X_test = clean_features(X_test_raw)

print("Training features shape:", X.shape)
print("Test features shape:", X_test.shape)
print("Target classes:", sorted(y.unique()))

Training features shape: (100000, 23)
Test features shape: (50000, 23)
Target classes: ['Good', 'Poor', 'Standard']


## 5. Identify numerical and categorical columns

In [7]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", len(numeric_features), numeric_features)
print("Categorical features:", len(categorical_features), categorical_features)

Numeric features: 16 ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance']
Categorical features: 7 ['Month', 'Occupation', 'Type_of_Loan', 'Credit_Mix', 'Credit_History_Age', 'Payment_of_Min_Amount', 'Payment_Behaviour']


## 6. Split training data

In [8]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

X_train: (80000, 23)
X_valid: (20000, 23)


## 7. Build preprocessing pipeline

In [9]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


## 8. Train and compare models

In [10]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

trained_models = {}
results = []

for model_name, classifier in models.items():
    model_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    print(f"Training {model_name}...")
    model_pipeline.fit(X_train, y_train)
    valid_predictions = model_pipeline.predict(X_valid)

    trained_models[model_name] = model_pipeline
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_valid, valid_predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_valid, valid_predictions),
        "Precision (weighted)": precision_score(y_valid, valid_predictions, average="weighted", zero_division=0),
        "Recall (weighted)": recall_score(y_valid, valid_predictions, average="weighted", zero_division=0),
        "F1 Score (weighted)": f1_score(y_valid, valid_predictions, average="weighted", zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1 Score (weighted)", ascending=False)
display(results_df)

Training Random Forest...
Training Logistic Regression...


,Model,Accuracy,Balanced Accuracy,Precision (weighted),Recall (weighted),F1 Score (weighted)
0,Random Forest,0.66195,0.709467,0.718930,0.66195,0.666398
1,Logistic Regression,0.65205,0.690039,0.690824,0.65205,0.655714


## 9. Evaluate the best model

In [11]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
valid_predictions = best_model.predict(X_valid)

print("Best model:", best_model_name)
print("\\nClassification report:")
print(classification_report(y_valid, valid_predictions, zero_division=0))

print("Confusion matrix (class order shown below):")
print(sorted(y.unique()))
print(confusion_matrix(y_valid, valid_predictions, labels=sorted(y.unique())))

Best model: Random Forest
\nClassification report:
              precision    recall  f1-score   support

        Good       0.48      0.82      0.60      3566
        Poor       0.64      0.74      0.69      5799
    Standard       0.84      0.56      0.68     10635

    accuracy                           0.66     20000
   macro avg       0.65      0.71      0.66     20000
weighted avg       0.72      0.66      0.67     20000

Confusion matrix (class order shown below):
['Good', 'Poor', 'Standard']
[[2927   83  556]
 [ 936 4310  553]
 [2273 2360 6002]]


## 10. Save the trained model

In [12]:
OUTPUT_DIR = "/kaggle/working"
MODEL_DIR = os.path.join(OUTPUT_DIR, "model")
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "credit_score_model.pkl")
joblib.dump(best_model, MODEL_PATH)

print("Best model saved successfully!")
print("Model name:", best_model_name)
print("Model path:", MODEL_PATH)
print("File exists:", os.path.exists(MODEL_PATH))

Best model saved successfully!
Model name: Random Forest
Model path: /kaggle/working/model/credit_score_model.pkl
File exists: True


## 11. Generate predictions for test data

In [13]:
# Ensure test features have exactly the same columns and order as training features.
X_test = X_test.reindex(columns=X.columns)

final_predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "Predicted_Credit_Score": final_predictions
})

PREDICTION_PATH = os.path.join(OUTPUT_DIR, "credit_score_predictions.csv")
submission.to_csv(PREDICTION_PATH, index=False)

print("FINAL PREDICTIONS COMPLETED!")
print("Total predictions:", len(submission))
print("\\nPrediction distribution:")
display(submission["Predicted_Credit_Score"].value_counts())
print("\\nFirst 10 predictions:")
display(submission.head(10))
print("Prediction file saved:", PREDICTION_PATH)

FINAL PREDICTIONS COMPLETED!
Total predictions: 50000
\nPrediction distribution:


Predicted_Credit_Score
Standard    17442
Poor        17245
Good        15313
Name: count, dtype: int64

\nFirst 10 predictions:


,ID,Predicted_Credit_Score
0,0x160a,Good
1,0x160b,Good
2,0x160c,Good
3,0x160d,Good
4,0x1616,Good
5,0x1617,Good
6,0x1618,Good
7,0x1619,Good
8,0x1622,Good
9,0x1623,Good


Prediction file saved: /kaggle/working/credit_score_predictions.csv


## 12. Final project files

In [14]:
print("Files created in /kaggle/working:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    for filename in files:
        print(os.path.join(root, filename))

Files created in /kaggle/working:
/kaggle/working/credit_score_predictions.csv
/kaggle/working/.virtual_documents/__notebook_source__.ipynb
/kaggle/working/model/credit_score_model.pkl


## Conclusion

The Credit Scoring Model project trains and evaluates two classification algorithms and selects the model with the higher weighted F1 score on a held-out validation set. The selected model is saved as `model/credit_score_model.pkl`, and predictions for the test dataset are saved as `credit_score_predictions.csv`.

**Note:** Validation scores can differ slightly from earlier runs depending on preprocessing details and library versions. This notebook is a reproducible rebuild of the workflow, not a recovery of the deleted original notebook.